In [15]:
import fitz
import os
import glob

In [4]:
base = "../RAG 2.0"

folders = [
    f"{base}/extracted",
    f"{base}/extracted/text",
    f"{base}/extracted/tables",
    f"{base}/extracted/images",
    f"{base}/extracted/page_labels",
    f"{base}/extracted/metadata",
    f"{base}/cleaned",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✔ Folder structure created successfully.")


✔ Folder structure created successfully.


In [42]:
def extract_page_labels(pdf_path, output_dir):
    import fitz, json, os
    os.makedirs(output_dir, exist_ok=True)

    doc = fitz.open(pdf_path)
    pdf_name = os.path.basename(pdf_path).replace(".pdf", "")
    labels = []

    for page in doc:
        page_num = page.number + 1
        label_raw = page.get_label()
        is_roman = label_raw.isdigit() is False if label_raw else False
        numeric = int(label_raw) if label_raw.isdigit() else None

        labels.append({
            "page": page_num,
            "raw": label_raw,
            "numeric": numeric,
            "is_roman": is_roman
        })

    out = os.path.join(output_dir, f"{pdf_name}_page_labels.json")
    with open(out, "w") as f:
        json.dump({"file": pdf_name, "labels": labels}, f, indent=2)

    return labels


In [53]:
import fitz, json, os

def clean_for_json(obj):
    if isinstance(obj, bytes):
        try:
            return obj.decode("utf-8", errors="ignore")
        except:
            return str(obj)

    if isinstance(obj, (tuple, set)):
        return list(obj)

    if isinstance(obj, dict):
        return {k: clean_for_json(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [clean_for_json(v) for v in obj]

    return obj


def extract_text_blocks(pdf_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)

    pdf_name = os.path.basename(pdf_path).replace(".pdf", "")

    for page in doc:
        page_num = page.number + 1

        blocks = page.get_text("dict")["blocks"]
        blocks = clean_for_json(blocks)

        output = {
            "file": pdf_name,
            "page": page_num,
            "blocks": blocks
        }

        out_path = os.path.join(output_dir, f"{pdf_name}_blocks_page_{page_num:03}.json")

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2)


In [44]:
def extract_tables(pdf_path, output_dir):
    import os, json, fitz

    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    pdf_name = os.path.basename(pdf_path).replace(".pdf", "")

    for page in doc:
        page_num = page.number + 1
        table_finder = page.find_tables()

        page_tables = {"file": pdf_name, "page": page_num, "tables": []}

        if table_finder:
            for t in table_finder.tables:
                page_tables["tables"].append({"bbox": t.bbox, "data": t.extract()})

        out = os.path.join(output_dir, f"{pdf_name}_tables_page_{page_num:03}.json")
        with open(out, "w") as f:
            json.dump(page_tables, f, indent=2)


In [50]:
def extract_metadata(pdf_path, output_dir):
    import os, json, fitz
    os.makedirs(output_dir, exist_ok=True)

    doc = fitz.open(pdf_path)
    pdf_name = os.path.basename(pdf_path).replace(".pdf", "")

    labels = extract_page_labels(pdf_path, output_dir)

    label_map = {l["page"]: l for l in labels}

    metadata = {"file": pdf_name, "total_pages": len(doc), "pages": []}

    for page in doc:
        page_num = page.number + 1
        label = label_map.get(page_num, {"raw": None, "numeric": None, "is_roman": False})

        text_dict = page.get_text("dict")
        fonts, sizes = set(), set()
        block_count = 0

        for b in text_dict["blocks"]:
            if "lines" in b:
                block_count += 1
                for line in b["lines"]:
                    for span in line["spans"]:
                        fonts.add(span["font"])
                        sizes.add(span["size"])

        metadata["pages"].append({
            "page": page_num,
            "label_raw": label["raw"],
            "label_numeric": label["numeric"],
            "is_roman": label["is_roman"],
            "font_inventory": sorted(list(fonts)),
            "font_sizes": sorted(list(sizes)),
            "num_blocks": block_count,
            "num_images": len(page.get_images(full=True)),
            "num_tables": len(page.find_tables().tables) if page.find_tables() else 0
        })

    out = os.path.join(output_dir, f"{pdf_name}_metadata.json")
    with open(out, "w") as f:
        json.dump(metadata, f, indent=2)


In [51]:
def process_all_pdfs(base_folder="../RAG 2.0"):
    """
    Master pipeline to process every PDF inside raw_pdfs/.
    Keeps everything modular and clean.
    """

    raw_folder = f"{base_folder}/raw_data"

    # Output folders
    out_text = f"{base_folder}/extracted/text"
    out_tables = f"{base_folder}/extracted/tables"
    out_images = f"{base_folder}/extracted/images"
    out_blocks = f"{base_folder}/extracted/blocks"
    out_labels = f"{base_folder}/extracted/page_labels"
    out_metadata = f"{base_folder}/extracted/metadata"

    # Create output dirs
    for folder in [out_text, out_tables, out_images, out_blocks, out_labels, out_metadata]:
        os.makedirs(folder, exist_ok=True)

    # List PDFs
    pdf_files = glob.glob(raw_folder + "/*.pdf")

    if not pdf_files:
        print("❌ No PDFs found in raw_pdfs folder.")
        return

    print(f"\nFound {len(pdf_files)} PDF files.\n")

    # -----------------------------
    # PROCESS EACH PDF
    # -----------------------------
    for pdf in pdf_files:
        print("\n====================================")
        print("Processing:", pdf)
        print("====================================")

        # 1) text blocks (with fonts, bbox, headings)
        extract_text_blocks(pdf, out_blocks)

        # 2) plain extracted text
        extract_text_blocks(pdf, out_text)

        # 3) tables
        extract_tables(pdf, out_tables)

        # 4) images
        extract_images(pdf, out_images)

        # 5) page labels (roman vs normal)
        extract_page_labels(pdf, out_labels)

        # 6) metadata summary
        extract_metadata(pdf, out_metadata)

    print("\n🎉 ALL PDFs processed successfully!")

In [54]:
process_all_pdfs()


Found 10 PDF files.


Processing: ../RAG 2.0/raw_data\common childhood.pdf
Consider using the pymupdf_layout package for a greatly improved page layout analysis.

Processing: ../RAG 2.0/raw_data\Guidance for national.pdf

Processing: ../RAG 2.0/raw_data\Latent.pdf

Processing: ../RAG 2.0/raw_data\module1.pdf

Processing: ../RAG 2.0/raw_data\module2.pdf

Processing: ../RAG 2.0/raw_data\module3.pdf

Processing: ../RAG 2.0/raw_data\module4.pdf

Processing: ../RAG 2.0/raw_data\module5.pdf

Processing: ../RAG 2.0/raw_data\module6.pdf

Processing: ../RAG 2.0/raw_data\Recommendations.pdf


FzErrorArgument: code=4: pixmap must be grayscale or rgb to write as png